# Animal Detector Trainer
This notebook extracts frames from videos, prepares a YOLO dataset, trains/validates a detector, and computes center points from detections.

## Step 1: Setup and paths
1. Set the folder that holds your videos.
2. Choose how many frames you want per video.
3. Run the cell to install and import dependencies.

In [ ]:
# Install dependencies
%pip -q install ultralytics opencv-python

# Imports
from pathlib import Path
import os
import random
import shutil
import time
import yaml
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
# Config
video_folder = Path("/path/to/videos")  # Folder containing your videos
frames_per_video = 25  # X photos per video
image_ext = ".jpg"
random_seed = 42
train_split = 0.8

# Output dataset folders
dataset_root = Path("/path/to/dataset")
images_train_dir = dataset_root / "images" / "train"
images_val_dir = dataset_root / "images" / "val"
labels_train_dir = dataset_root / "labels" / "train"
labels_val_dir = dataset_root / "labels" / "val"

for d in [images_train_dir, images_val_dir, labels_train_dir, labels_val_dir]:
    d.mkdir(parents=True, exist_ok=True)

## Step 2: Extract frames from videos
1. Put your videos in the folder you set above.
2. Run the next cell to extract frames into per-video subfolders.
3. Annotate those frames with your labeling tool (YOLO format).

In [ ]:
def extract_frames_from_video(video_path: Path, frames_count: int, out_dir: Path, ext: str = ".jpg"):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Skipping {video_path.name}: cannot open")
        return 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        print(f"Skipping {video_path.name}: no frames")
        cap.release()
        return 0
    out_dir.mkdir(parents=True, exist_ok=True)
    indices = np.linspace(0, total_frames - 1, frames_count, dtype=int)
    saved = 0
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            continue
        out_file = out_dir / f"{video_path.stem}_f{idx:06d}{ext}"
        cv2.imwrite(str(out_file), frame)
        saved += 1
    cap.release()
    return saved

video_files = sorted([p for p in video_folder.iterdir() if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}])
if not video_files:
    print("No videos found. Check video_folder.")
else:
    for video_path in video_files:
        out_dir = video_path.parent / f"{video_path.stem}_frames"
        saved = extract_frames_from_video(video_path, frames_per_video, out_dir, image_ext)
        print(f"{video_path.name}: saved {saved} frames to {out_dir}")

## Step 3: Prepare train/val folders
1. Collect your labeled frames and YOLO label files in one folder.
2. Set `labeled_root` to that folder and run the next cell.
3. You will get `images/train`, `images/val`, `labels/train`, `labels/val`.

In [ ]:
# Folder that contains both images and YOLO .txt labels with matching names
labeled_root = Path("/path/to/labeled_frames")

# Class names in your dataset (order must match label IDs)
class_names = ["animal"]

def list_image_label_pairs(root: Path, exts={".jpg", ".jpeg", ".png"}):
    pairs = []
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() in exts:
            label_path = img_path.with_suffix(".txt")
            if label_path.exists():
                pairs.append((img_path, label_path))
    return pairs

pairs = list_image_label_pairs(labeled_root)
if not pairs:
    raise RuntimeError("No image/label pairs found. Check labeled_root.")

random.seed(random_seed)
random.shuffle(pairs)
split_idx = int(len(pairs) * train_split)
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

def copy_pairs(pairs_list, images_dir: Path, labels_dir: Path):
    for img_path, label_path in pairs_list:
        shutil.copy2(img_path, images_dir / img_path.name)
        shutil.copy2(label_path, labels_dir / label_path.name)

copy_pairs(train_pairs, images_train_dir, labels_train_dir)
copy_pairs(val_pairs, images_val_dir, labels_val_dir)

data_yaml = {
    "path": str(dataset_root),
    "train": "images/train",
    "val": "images/val",
    "nc": len(class_names),
    "names": class_names,
}
with open(dataset_root / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Train images: {len(train_pairs)} | Val images: {len(val_pairs)}")
print(f"Wrote {dataset_root / 'data.yaml'}")

## Step 4: Train and validate
1. Choose a base model or let the notebook pick the most recent run.
2. Run training and validation on your new dataset.
3. Your best weights are saved under `runs/detect/train*/weights/`.

In [ ]:
def find_latest_weights(search_dir: Path) -> Path | None:
    candidates = list(search_dir.rglob("*/weights/best.pt")) + list(search_dir.rglob("*/weights/last.pt"))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

runs_dir = Path("runs") / "detect"
latest_weights = find_latest_weights(runs_dir)
base_model = latest_weights if latest_weights else "yolov8n.pt"
print(f"Using base model: {base_model}")

model = YOLO(str(base_model))
train_results = model.train(
    data=str(dataset_root / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    project="runs/detect",
    name="train_custom",
    exist_ok=True,
)

val_results = model.val(data=str(dataset_root / "data.yaml"))
print(val_results)

## Step 5: Get center pixel coordinates
1. Pick an image to test your detector.
2. Run inference with the trained weights.
3. The next cell prints $(x, y)$ centers for each detection.

In [ ]:
# Use your latest trained weights for inference
inference_weights = find_latest_weights(runs_dir) or base_model
infer_model = YOLO(str(inference_weights))

test_image = Path("/path/to/test_image.jpg")
results = infer_model.predict(source=str(test_image), imgsz=640, conf=0.25, verbose=False)

centers = []
for r in results:
    if r.boxes is None:
        continue
    for box in r.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        centers.append((float(cx), float(cy)))

print("Centers (x, y):")
for c in centers:
    print(c)

## Step 6: Measure distance traveled
1. Set a folder of ordered frames and (optionally) an output CSV path.
2. Run detection across frames to compute center points and distances.
3. Review the printed table or saved CSV for totals and per-frame stats.

In [ ]:
# Folder of frames in order (e.g., a *_frames folder or your val/train folder)
frames_folder = Path("/path/to/frames_folder")
output_csv = Path("/path/to/output.csv")  # Set to None to skip saving
target_class_id = 0
conf_threshold = 0.25
pixels_per_unit = None  # Set to e.g. 10.0 if you want distance in real-world units

# Ensure we have a model for inference
if "infer_model" not in globals():
    inference_weights = find_latest_weights(runs_dir) or base_model
    infer_model = YOLO(str(inference_weights))

def get_center_for_frame(img_path: Path):
    results = infer_model.predict(source=str(img_path), imgsz=640, conf=conf_threshold, verbose=False)
    best = None
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        boxes = r.boxes
        cls = boxes.cls.cpu().numpy().astype(int)
        conf = boxes.conf.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        for i in range(len(cls)):
            if cls[i] != target_class_id:
                continue
            if best is None or conf[i] > best["conf"]:
                x1, y1, x2, y2 = xyxy[i]
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                best = {"cx": float(cx), "cy": float(cy), "conf": float(conf[i])}
    return best

frame_paths = sorted([p for p in frames_folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
if not frame_paths:
    raise RuntimeError("No frames found. Check frames_folder.")

rows = []
prev = None
cum_dist = 0.0
for idx, img_path in enumerate(frame_paths):
    center = get_center_for_frame(img_path)
    if center is None:
        rows.append({
            "frame": img_path.name,
            "frame_index": idx,
            "center_x": None,
            "center_y": None,
            "confidence": None,
            "distance_from_prev": None,
            "cumulative_distance": cum_dist,
        })
        continue
    dist = None
    if prev is not None:
        dx = center["cx"] - prev["cx"]
        dy = center["cy"] - prev["cy"]
        dist = float((dx ** 2 + dy ** 2) ** 0.5)
        if pixels_per_unit:
            dist = dist / float(pixels_per_unit)
        cum_dist += dist
    rows.append({
        "frame": img_path.name,
        "frame_index": idx,
        "center_x": center["cx"],
        "center_y": center["cy"],
        "confidence": center["conf"],
        "distance_from_prev": dist,
        "cumulative_distance": cum_dist,
    })
    prev = center

# Display table (pandas if available, otherwise print rows)
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
    if output_csv:
        df.to_csv(output_csv, index=False)
        print(f"Saved CSV to {output_csv}")
except Exception:
    for row in rows[:5]:
        print(row)
    if output_csv:
        import csv
        with open(output_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
        print(f"Saved CSV to {output_csv}")